[Reference](https://anill-hayriye.medium.com/exploratory-analysis-for-time-series-ad821abb517e)

In [4]:
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from plotly.subplots import make_subplots

seasonal_decomposition = seasonal_decompose(dataset['temperature_2m (°C)'], model='additive', period=24*365)
trend = seasonal_decomposition.trend
seasonal = seasonal_decomposition.seasonal
residual = seasonal_decomposition.resid

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Observed Data", "Trend", "Seasonal", "Residual")
)

fig.add_trace(go.Scatter(x=dataset.index, y=dataset['temperature_2m (°C)'], mode='lines', name='Observed', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=dataset.index, y=trend, mode='lines', name='Trend', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=dataset.index, y=seasonal, mode='lines', name='Seasonal', line=dict(color='green')), row=3, col=1)
fig.add_trace(go.Scatter(x=dataset.index, y=residual, mode='lines', name='Residual', line=dict(color='red')), row=4, col=1)

fig.update_layout(
    title="Seasonal Decomposition of Temperature Data",
    xaxis_title="Date",
    yaxis_title="Temperature (°C)",
    height=800,
    showlegend=False
)
fig.show()

# Stationary

In [5]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(dataset["temperature_2m (°C)"])
labels = ['ADF Test Statistic', 'p-value', '#Lags Used', 'Number of Observations']

for value, label in zip(result, labels):
    print(f"{label}: {value}")

if result[1] <= 0.05:
    print("Data is stationary")
else:
    print("Data is not stationary")

In [6]:
# First order differencing
data_frame["temperature_2m (°C)_diff_1"] = dataset["temperature_2m (°C)"].diff(1)
df_diff = data_frame['temperature_2m (°C)_diff_1'].dropna()
df_diff

In [7]:
import numpy as np

# Log transformation
data_frame['temperature_log'] = np.log(dataset["temperature_2m (°C)"])

# Square root transformation
data_frame['temperature_sqrt'] = np.sqrt(dataset["temperature_2m (°C)"])

# You can combine with differencing:
data_frame['log_diff'] = data_frame['temperature_log'].diff()
df_log_diff = data_frame['log_diff'].dropna()
df_log_diff

In [8]:
seasonal_lag = 24 * 365
data_frame['seasonal_diff'] = data_frame['temperature_2m (°C)'] - data_frame['temperature_2m (°C)'].shift(seasonal_lag)
df_seasonal_diff = data_frame['seasonal_diff'].dropna()
df_seasonal_diff

# Autocorrelation Function (ACF)

In [9]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(10, 5))
plot_acf(dataset["temperature_2m (°C)"], lags=24)
plt.title("Autocorrelation Function (ACF) of Temperature")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.show()